NLTK의 문장 토큰화, 단어 토큰화, 불용어 목록을 사용하기 위해 필요한 도구를 불러옵니다.


In [4]:
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

정수 인코딩 예제에 사용할 원문 텍스트를 하나의 문자열로 준비합니다.


In [5]:
raw_text = "A barber is a person. a barber is good person. a barber is huge person. he Knew A Secret! The Secret He Kept is huge secret. Huge secret. His barber kept his word. a barber kept his word. His barber kept his secret. But keeping and keeping such a huge secret to himself was driving the barber crazy. the barber went up a huge mountain."

원문을 문장 단위로 나누어 이후 단어 전처리를 문장별로 수행할 수 있게 만듭니다.


In [6]:
# 문장 토큰화
sentences = sent_tokenize(raw_text)
print(sentences)

['A barber is a person.', 'a barber is good person.', 'a barber is huge person.', 'he Knew A Secret!', 'The Secret He Kept is huge secret.', 'Huge secret.', 'His barber kept his word.', 'a barber kept his word.', 'His barber kept his secret.', 'But keeping and keeping such a huge secret to himself was driving the barber crazy.', 'the barber went up a huge mountain.']


각 문장을 단어로 나눈 뒤 소문자화, 불용어 제거, 짧은 단어 제거를 적용하고 단어 빈도를 집계합니다.


In [7]:
vocab = {}
preprocessed_sentences = []
stop_words = set(stopwords.words('english'))

for sentence in sentences:
    # 단어 토큰화
    tokenized_sentence = word_tokenize(sentence)
    result = []

    for word in tokenized_sentence:
        word = word.lower() # 모든 단어를 소문자화하여 단어의 개수를 줄인다.
        if word not in stop_words: # 단어 토큰화 된 결과에 대해서 불용어를 제거한다.
            if len(word) > 2: # 단어 길이가 2이하인 경우에 대하여 추가로 단어를 제거한다.
                result.append(word)
                if word not in vocab:
                    vocab[word] = 0
                vocab[word] += 1

    preprocessed_sentences.append(result)
print(preprocessed_sentences)

[['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]


전처리된 단어들이 전체 말뭉치에서 몇 번 등장했는지 확인합니다.


In [8]:
print('단어 집합 :', vocab)

단어 집합 : {'barber': 8, 'person': 3, 'good': 1, 'huge': 5, 'knew': 1, 'secret': 6, 'kept': 4, 'word': 2, 'keeping': 2, 'driving': 1, 'crazy': 1, 'went': 1, 'mountain': 1}


`barber`라는 단어가 전처리된 말뭉치에서 몇 번 등장했는지 빈도값을 직접 조회합니다.


In [9]:
# 'barber'라는 단어의 빈도수 출력
print(vocab["barber"])

8


단어 빈도 사전을 등장 횟수가 많은 순서대로 정렬합니다.


In [10]:
vocab_sorted = sorted(vocab.items(), key = lambda x:x[1], reverse=True)
print(vocab_sorted)

[('barber', 8), ('secret', 6), ('huge', 5), ('kept', 4), ('person', 3), ('word', 2), ('keeping', 2), ('good', 1), ('knew', 1), ('driving', 1), ('crazy', 1), ('went', 1), ('mountain', 1)]


빈도가 2회 이상인 단어에만 정수 인덱스를 부여해 기본 단어 집합을 만듭니다.


In [11]:
word_to_index = {}
i = 0
for (word, frequency) in vocab_sorted :
    if frequency > 1 : # 빈도수가 작은 단어는 제외.
        i = i + 1
        word_to_index[word] = i

print(word_to_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5, 'word': 6, 'keeping': 7}


사용할 단어 집합 크기를 제한하고 순위가 낮은 단어의 인덱스를 제거합니다.


In [12]:
vocab_size = 5

# 인덱스가 5 초과인 단어 제거
words_frequency = [word for word, index in word_to_index.items() if index >= vocab_size + 1]

# 해당 단어에 대한 인덱스 정보를 삭제
for w in words_frequency:
    del word_to_index[w]
print(word_to_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5}


단어 집합에 없는 단어를 표현하기 위한 OOV 인덱스를 마지막에 추가합니다.


In [13]:
word_to_index['OOV'] = len(word_to_index) + 1
print(word_to_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5, 'OOV': 6}


전처리된 문장의 각 단어를 정수 인덱스로 바꾸고, 미등록 단어는 OOV 인덱스로 대체합니다.


In [14]:
encoded_sentences = []
for sentence in preprocessed_sentences:
    encoded_sentence = []
    for word in sentence:
        try:
            # 단어 집합에 있는 단어라면 해당 단어의 정수를 리턴.
            encoded_sentence.append(word_to_index[word])
        except KeyError:
            # 만약 단어 집합에 없는 단어라면 'OOV'의 정수를 리턴.
            encoded_sentence.append(word_to_index['OOV'])
    encoded_sentences.append(encoded_sentence)
print(encoded_sentences)


[[1, 5], [1, 6, 5], [1, 3, 5], [6, 2], [2, 4, 3, 2], [3, 2], [1, 4, 6], [1, 4, 6], [1, 4, 2], [6, 6, 3, 2, 6, 1, 6], [1, 6, 3, 6]]


`Counter`를 사용해 리스트에 담긴 단어들의 등장 횟수를 간단하게 계산합니다.

In [15]:
from collections import Counter

정수 인코딩에 사용할 전처리 결과가 문장별 단어 리스트 형태인지 확인합니다.

In [16]:
print(preprocessed_sentences)

[['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]


문장 단위로 나뉜 단어 리스트를 하나의 단어 리스트로 평탄화합니다.

In [17]:
# words = np.hstack(preprocessed_senteces) 으로도 수행 가능.
all_words_list = sum(preprocessed_sentences, [])
print(all_words_list)

['barber', 'person', 'barber', 'good', 'person', 'barber', 'huge', 'person', 'knew', 'secret', 'secret', 'kept', 'huge', 'secret', 'huge', 'secret', 'barber', 'kept', 'word', 'barber', 'kept', 'word', 'barber', 'kept', 'secret', 'keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy', 'barber', 'went', 'huge', 'mountain']


평탄화된 단어 리스트를 `Counter`에 넣어 단어 빈도 사전을 생성합니다.

In [18]:
# 파이썬의 Counter 모듈을 이용하여 단어의 빈도수 카운트
vocab = Counter(all_words_list)
print(vocab)

Counter({'barber': 8, 'secret': 6, 'huge': 5, 'kept': 4, 'person': 3, 'word': 2, 'keeping': 2, 'good': 1, 'knew': 1, 'driving': 1, 'crazy': 1, 'went': 1, 'mountain': 1})


생성된 빈도 사전에서 특정 단어의 등장 횟수를 직접 조회합니다.

In [19]:
print(vocab["barber"]) # 'barber' 라는 단어의 빈도수 출력

8


빈도수가 높은 상위 단어만 남겨 제한된 크기의 단어 집합을 만듭니다.

In [20]:
vocab_size = 5
vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 5개의 단어만 저장
vocab

[('barber', 8), ('secret', 6), ('huge', 5), ('kept', 4), ('person', 3)]

상위 빈도 단어에 1부터 시작하는 정수 인덱스를 순서대로 부여합니다.

In [21]:
word_to_index = {}
i = 0
for (word, frequency) in vocab:
    i = i + 1
    word_to_index[word] = i

print(word_to_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5}


NLTK의 `FreqDist`와 NumPy를 사용해 같은 빈도 기반 단어 집합 생성을 실습합니다.

In [22]:
from nltk import FreqDist
import numpy as np

`np.hstack`으로 문장 경계를 제거한 뒤 `FreqDist`로 단어 빈도를 계산합니다.

In [23]:
# np.hstack으로 문장 구분을 제거
vocab = FreqDist(np.hstack(preprocessed_sentences))

`FreqDist` 객체에서도 특정 단어의 빈도수를 동일하게 조회할 수 있습니다.

In [24]:
print(vocab["barber"]) # 'barber'라는 단어의 빈도수 출력

8


`FreqDist` 결과에서 등장 빈도가 높은 상위 5개 단어만 추립니다.

In [25]:
vocab_size = 5
vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 5개의 단어만 저장
print(vocab)

[(np.str_('barber'), 8), (np.str_('secret'), 6), (np.str_('huge'), 5), (np.str_('kept'), 4), (np.str_('person'), 3)]


딕셔너리 컴프리헨션으로 단어와 정수 인덱스의 매핑을 간결하게 생성합니다.

In [26]:
word_to_index = {word[0] : index + 1 for index, word in enumerate(vocab)}
print(word_to_index)

{np.str_('barber'): 1, np.str_('secret'): 2, np.str_('huge'): 3, np.str_('kept'): 4, np.str_('person'): 5}


`enumerate`가 값과 인덱스를 함께 반환하는 방식을 간단한 예제로 확인합니다.

In [27]:
test_input = ['a', 'b', 'c', 'd', 'e']
for index, value in enumerate(test_input): # 입력의 순서대로 0부터 인덱스를 부여함.
    print("value : {}, index: {}".format(value, index))

value : a, index: 0
value : b, index: 1
value : c, index: 2
value : d, index: 3
value : e, index: 4


Keras의 `Tokenizer`를 불러와 단어 집합 생성과 정수 인코딩을 자동화합니다.

In [28]:
from tensorflow.keras.preprocessing.text import Tokenizer

Tokenizer 실습에 사용할 전처리된 문장 리스트를 명시적으로 준비합니다.

In [29]:
preprocessed_sentences = [['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]


Tokenizer를 코퍼스에 학습시켜 단어 빈도와 단어-인덱스 정보를 내부에 저장합니다.

In [30]:
tokenizer = Tokenizer()

# fit_on_textxs()안에 코퍼스를 입력으로 하면 빈도수를 기준으로 단어 집합을 생성.
tokenizer.fit_on_texts(preprocessed_sentences)

Tokenizer가 빈도 기준으로 만든 단어별 정수 인덱스를 확인합니다.

In [31]:
print(tokenizer.word_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5, 'word': 6, 'keeping': 7, 'good': 8, 'knew': 9, 'driving': 10, 'crazy': 11, 'went': 12, 'mountain': 13}


Tokenizer가 계산한 각 단어의 등장 횟수를 확인합니다.

In [32]:
print(tokenizer.word_counts)

OrderedDict([('barber', 8), ('person', 3), ('good', 1), ('huge', 5), ('knew', 1), ('secret', 6), ('kept', 4), ('word', 2), ('keeping', 2), ('driving', 1), ('crazy', 1), ('went', 1), ('mountain', 1)])


학습된 단어 인덱스를 이용해 문장별 단어 시퀀스를 정수 시퀀스로 변환합니다.

In [33]:
print(tokenizer.texts_to_sequences(preprocessed_sentences))

[[1, 5], [1, 8, 5], [1, 3, 5], [9, 2], [2, 4, 3, 2], [3, 2], [1, 4, 6], [1, 4, 6], [1, 4, 2], [7, 7, 3, 2, 10, 1, 11], [1, 12, 3, 13]]


`num_words` 옵션으로 실제 인코딩에 사용할 상위 빈도 단어 수를 제한합니다.

In [34]:
vocab_size = 5
tokenizer = Tokenizer(num_words=vocab_size + 1) # 상위 5개 단어만 사용
tokenizer.fit_on_texts(preprocessed_sentences)

제한 옵션을 두어도 내부 단어 인덱스는 전체 단어 기준으로 저장되는지 확인합니다.

In [35]:
print(tokenizer.word_index)

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5, 'word': 6, 'keeping': 7, 'good': 8, 'knew': 9, 'driving': 10, 'crazy': 11, 'went': 12, 'mountain': 13}


Tokenizer가 보관하는 전체 단어 빈도 정보를 함께 확인합니다.

In [36]:
print(tokenizer.word_counts)

OrderedDict([('barber', 8), ('person', 3), ('good', 1), ('huge', 5), ('knew', 1), ('secret', 6), ('kept', 4), ('word', 2), ('keeping', 2), ('driving', 1), ('crazy', 1), ('went', 1), ('mountain', 1)])


정수 시퀀스로 변환할 때는 `num_words` 범위 안의 단어만 남는지 확인합니다.

In [37]:
print(tokenizer.texts_to_sequences(preprocessed_sentences))

[[1, 5], [1, 5], [1, 3, 5], [2], [2, 4, 3, 2], [3, 2], [1, 4], [1, 4], [1, 4, 2], [3, 2, 1], [1, 3]]


단어 집합을 직접 수정해보기 위해 제한 옵션 없이 Tokenizer를 다시 학습합니다.

In [38]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(preprocessed_sentences)

인덱스 순위가 낮은 단어를 직접 삭제해 원하는 크기의 단어 집합만 남깁니다.

In [39]:
vocab_size = 5
words_frequency = [word for word, index in tokenizer.word_index.items() if index >= vocab_size + 1]

# 인덱스가 5 초과인 단어 제거
for word in words_frequency:
    del tokenizer.word_index[word] # 해당 단어에 대한 인덱스 정보를 삭제
    del tokenizer.word_counts[word] # 해당 단어에 대한 카운트 정보를 삭제

print(tokenizer.word_index)
print(tokenizer.word_counts)
print(tokenizer.texts_to_sequences(preprocessed_sentences))

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5}
OrderedDict([('barber', 8), ('person', 3), ('huge', 5), ('secret', 6), ('kept', 4)])
[[1, 5], [1, 5], [1, 3, 5], [2], [2, 4, 3, 2], [3, 2], [1, 4], [1, 4], [1, 4, 2], [3, 2, 1], [1, 3]]


패딩용 0과 미등록 단어 처리를 고려해 OOV 토큰이 포함된 Tokenizer를 설정합니다.

In [40]:
# 숫자 0과 OOV를 고려해서 단어 집합의 크기는 +2
vocab_size = 5
tokenizer = Tokenizer(num_words = vocab_size + 2, oov_token = 'OOV')
tokenizer.fit_on_texts(preprocessed_sentences)


OOV 토큰이 실제로 어떤 정수 인덱스를 부여받았는지 확인합니다.

In [41]:
print('단어 OOV의 인덱스 : {}'.format(tokenizer.word_index['OOV']))


단어 OOV의 인덱스 : 1


단어 집합 밖의 단어가 OOV 인덱스로 대체되어 인코딩되는지 확인합니다.

In [42]:
print(tokenizer.texts_to_sequences(preprocessed_sentences))


[[2, 6], [2, 1, 6], [2, 4, 6], [1, 3], [3, 5, 4, 3], [4, 3], [2, 5, 1], [2, 5, 1], [2, 5, 3], [1, 1, 4, 3, 1, 2, 1], [2, 1, 4, 1]]
